# Financial Analysis

Computes annual network cost from actual consumption and the official tariff, and answers the
question: *"if the highest peak were reduced by X%, how much would this customer save?"*

This runs on the **single test customer** generated in `00_synthetic_customer_generator.ipynb` and processed in `02_peaks_and_features.ipynb` - not the bulk training population, which exists only to train K-Means (see `03_kmeans_clustering.ipynb`). Loading that notebook's precomputed features (rather than generating a new customer here) keeps this notebook and `05_llm_profile_diagnostic.ipynb` looking at the same customer, and avoids recomputing peaks/features that notebook 02 already produced.

## Scope

- **Electricity only.** Only Westnetz/Amprion electricity tariff PDFs have been collected so far -
  no gas pricing exists yet.
- **Network charges only** (Leistungspreis + Arbeitspreis, single tariff profile) - no surcharges,
  no concession fees, no energy supply cost. Matches the scope already established in `pricing.py`.

## Methodology

**Demand-charge bracket selection.** The German "Jahresleistungspreissystem" has two rate brackets,
chosen by the customer's own **utilization hours** (`annual kWh / peak kW`) - not a fixed choice.
Below 2500 h/a: lower demand charge, higher energy charge (favors peaky customers). At or above
2500 h/a: much higher demand charge, much lower energy charge (favors flat, high-utilization
customers). This bracket is re-selected any time peak changes, including in the reduction scenarios
below - a big enough peak cut can push a customer across the threshold into a **structurally
different** cost regime, not just a smaller version of the same one.

**Peak reduction scenarios.** Assume annual energy (kWh) stays constant when peak is reduced - this
models peak *shaving/shifting* (the peak moves elsewhere, e.g. via a battery or load shifting), not
an efficiency improvement that would also cut total energy.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")
import synthetic_generator as sg
import peaks as pk
import features as feat
import pricing as pr
import financial as fin
import pandas as pd
import matplotlib.pyplot as plt

OUTPUTS_DIR = Path("../data/outputs")
FINANCIAL_DIR = OUTPUTS_DIR / "financial"
FINANCIAL_DIR.mkdir(parents=True, exist_ok=True)

tariffs = pd.read_csv(OUTPUTS_DIR / "pricing" / "network_tariffs_normalized.csv")
default_tariff = pr.get_default_tariff(tariffs)
print("Tariff used:", default_tariff["voltage_level"], f"({default_tariff['operator']})")


## Load the single test customer

Loads the customer generated by `00_synthetic_customer_generator.ipynb` and its precomputed features from `02_peaks_and_features.ipynb`, via the `LATEST_CUSTOMER_ID.txt` pointer file those notebooks write - re-run 00 (and then 02) first for a different customer.

In [ ]:
SINGLE_TEST_DIR = Path("../data/inputs/generated/single_test")
CUSTOMER_ID = (SINGLE_TEST_DIR / "LATEST_CUSTOMER_ID.txt").read_text().strip()

customer_electricity = pd.read_csv(SINGLE_TEST_DIR / f"{CUSTOMER_ID}_electricity.csv")
customer = pd.read_csv(SINGLE_TEST_DIR / f"{CUSTOMER_ID}_electricity_features.csv")

print("customer_id:", CUSTOMER_ID)
display(customer)


## Current annual cost

In [ ]:
financials = fin.compute_customer_financials(customer, default_tariff)
display(financials)


## Cost breakdown

Demand charge (Leistungspreis) vs. energy charge (Arbeitspreis).

In [ ]:
row = financials.iloc[0]
plt.figure(figsize=(7, 4))
plt.barh(["Demand charge", "Energy charge"], [row["demand_charge_eur"], row["energy_charge_eur"]], color=["#d62728", "#2ca02c"])
plt.xlabel("EUR / year")
plt.title(f"Annual network cost breakdown - {CUSTOMER_ID}")
plt.tight_layout()
plt.show()


## Peak reduction scenarios - "what if the highest peak were X% lower?"

In [ ]:
scenario_rows = [fin.compute_scenario_for_all(customer, default_tariff, reduction_pct=pct) for pct in [5, 10, 15, 20, 30]]
scenarios = pd.concat(scenario_rows, ignore_index=True)
display(scenarios)

plt.figure(figsize=(8, 5))
plt.plot(scenarios["reduction_pct"], scenarios["savings_pct"], marker="o")
plt.xlabel("Assumed peak reduction (%)")
plt.ylabel("Cost savings (%)")
plt.title(f"Peak reduction sensitivity - {CUSTOMER_ID}")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

crossed = scenarios[scenarios["bracket_changed"]]
if not crossed.empty:
    print(f"Bracket changes at: {crossed['reduction_pct'].tolist()}% reduction")


## Save outputs

In [ ]:
financials_path = FINANCIAL_DIR / "electricity_financials.csv"
scenarios_path = FINANCIAL_DIR / "electricity_peak_reduction_scenarios.csv"

financials.to_csv(financials_path, index=False)
scenarios.to_csv(scenarios_path, index=False)

print("Saved:")
print(" ", financials_path)
print(" ", scenarios_path)
